In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np
import sys
import os
from netCDF4 import Dataset, date2num
import glob
import scipy.io
import tqdm
import xarray as xr
import os
import warnings
import matplotlib.pyplot as plt
import  scipy as sc
from pydsstools.heclib.dss import HecDss
from scipy.stats import pearsonr
import cartopy.feature as cfeature
import cartopy.crs as ccrs
from cartopy.feature import ShapelyFeature
from cartopy.io.shapereader import Reader
from sklearn import metrics
import seaborn as sns
import pickle
import spotpy
warnings.filterwarnings('ignore')

In [3]:
from osgeo import gdal, ogr
import os
import matplotlib.pyplot as plt
import tqdm
import shapely
from shapely.geometry import MultiPolygon, Point
from scipy.interpolate import griddata
from datetime import datetime
import glob
import os, shutil
from scipy import stats
import urllib.request
import time
from shapely.geometry.multipolygon import MultiPolygon
from shapely.geometry.polygon import Polygon
from shapely import wkt
from osgeo.gdalnumeric import *
from osgeo.gdalconst import *
from pandas.io.json import json_normalize
gdal.SetConfigOption("GDALWARP_IGNORE_BAD_CUTLINE", "YES")
import yaml

In [4]:
from funciones_cambio_climatico import *

In [5]:
modelos=[['CNRM-CERFACS-CNRM-CM5','SMHI-RCA4','r1i1p1'],
         ['CNRM-CERFACS-CNRM-CM5','KNMI-RACMO22E','r1i1p1'],
         ['ICHEC-EC-EARTH','SMHI-RCA4','r12i1p1'],
         ['ICHEC-EC-EARTH','KNMI-RACMO22E','r1i1p1'],
         ['ICHEC-EC-EARTH','KNMI-RACMO22E','r12i1p1'],
         ['ICHEC-EC-EARTH','DMI-HIRHAM5','r3i1p1'],
         ['ICHEC-EC-EARTH','CLMcom-CCLM4-8-17','r12i1p1'],
         ['IPSL-IPSL-CM5A-MR','SMHI-RCA4','r1i1p1'],
         ['IPSL-IPSL-CM5A-MR','IPSL-WRF381P','r1i1p1'],
         ['MOHC-HadGEM2-ES','SMHI-RCA4','r1i1p1'],
         ['MOHC-HadGEM2-ES','DMI-HIRHAM5','r1i1p1'],
         ['MOHC-HadGEM2-ES','CLMcom-CCLM4-8-17','r1i1p1'],
         ['MPI-M-MPI-ESM-LR','SMHI-RCA4','r1i1p1'],
         ['MPI-M-MPI-ESM-LR','CLMcom-CCLM4-8-17','r1i1p1'],
         ['NCC-NorESM1-M','DMI-HIRHAM5','r1i1p1']]

In [6]:
names_model=list()
for i in modelos:
    names_model.append(i[0]+'_'+i[1]+'_'+i[2])

In [7]:
name_columns = list()
for i in np.arange(1,13):
    name_columns.append(i)
    name_columns.append('Factor_'+str(i))

# Configuración del modelo

In [8]:
os.chdir('E:/Github/HEC-HMS/')

In [9]:
from funciones_HMS import *

In [10]:
Path_model ='E:/TUNEZ/Modelos_Nuevos/Oued_Mellah_CC/'
file_hms = 'Oued_Mellah_Python.hms'
file_gage = 'Oued_Mellah_Python.gage'
file_basin = 'Oued_Mellah_SED.basin'
file_run= 'Oued_Mellah_Python.run'

In [11]:
names_stations = read_gages(Path_model, file_gage)

FileNotFoundError: [Errno 2] No such file or directory: 'E:/TUNEZ/Modelos_Nuevos/Oued_Mellah_CC/Oued_Mellah_Python.gage'

In [ ]:
names_control = read_control(Path_model, file_hms)

In [ ]:
names_control

In [14]:
# NOMBRES DE FICHEROS -------------------------------------------------------------------------------------------------------------------------------------------------
Path_model ='E:/TUNEZ/Modelos_Nuevos/Oued_Mellah_C/'
path_rain = 'E:/TUNEZ/07_Cambio_Climatico/Archivo_temporal.csv' # El fichero debe contener una columna con fecha (index) y los datos de lluvias de estaciónes en cada columna 
name_model = 'Oued_Mellah_Python' # Nombre del fichero .hms sin la extensión
name_basin = 'Oued_Mellah_SED_CP' # Igual al arrojado en namesbasin
name_control = 'Campiazo' # Igual al arrojado en namecontrol
file_dss = 'Oued_Mellah_Python.dss' # Nombre de fichero donde se almacenan las lluvias

# DEFINICIÓN DEL TIEMPO E INTERVALOS PARA LLENAR LOS DATOS DE LLUVIAS EN EL PROGRAMA -------------------------------------------------------------------------------------------------
#Nota: Si el control ya esta generado en el programa, no es necesario llenar estos datos
Time_interval = '1DAY' #(1MIN, 2MIN, 3MIN,4MIN,5MIN,6MIN,10MIN,...,1HOUR,....1DAY) dependiendo de la resolución de los datos se puede variar
Time_interval_c ='60' # Se debe ingresar en minutos, en este caso se usa un dia = 60*24 = 1440

#Name_run = 'Run' # en caso de generar una nueva corrida se asigna el nombre que prefiera el usuario

In [15]:
Centroides = pd.read_csv('E:/TUNEZ/GIS/Centroides.csv',index_col=0)

In [16]:
names_stations =  Centroides.name.values

In [17]:
dict_CC_PREC_QM_CORDEX   =  pickle.load(open('E:/TUNEZ/07_Cambio_Climatico/CLIMA/Diccionarios/'+'dict_CC_PREC_QM_CORDEX.pickle','rb'))
dict_CC_TASMAX_QM_CORDEX =  pickle.load(open('E:/TUNEZ/07_Cambio_Climatico/CLIMA/Diccionarios/'+'dict_CC_TASMAX_QM_CORDEX.pickle','rb'))
dict_CC_TASMIN_QM_CORDEX =  pickle.load(open('E:/TUNEZ/07_Cambio_Climatico/CLIMA/Diccionarios/'+'dict_CC_TASMIN_QM_CORDEX.pickle','rb'))

In [18]:
sim = 0
for nmodel, m in enumerate(names_model):
    for rcp in ['rcp45','rcp85']:
        for per in [[2011,2040,'CP'],[2041,2070,'MP'],[2071,2100,'LP']]:
            if per[2] == 'CP' and rcp=='rcp45':
                name_basin = 'Oued_Mellah_SED_CP_RCP45'
                
            elif per[2] == 'CP' and rcp=='rcp85':
                name_basin = 'Oued_Mellah_SED_CP_RCP85'
                
            elif per[2] == 'MP' and rcp=='rcp45':
                name_basin = 'Oued_Mellah_SED_MP_RCP45'
                
            elif per[2] == 'MP' and rcp=='rcp85':
                name_basin = 'Oued_Mellah_SED_MP_RCP85'
                
            elif per[2] == 'LP' and rcp=='rcp45':
                name_basin = 'Oued_Mellah_SED_LP_RCP45'
                
            elif per[2] == 'LP' and rcp=='rcp85':
                name_basin = 'Oued_Mellah_SED_LP_RCP85'

            yearini = per[0]
            yearfin = per[1]
            
            names_stations = list()
            for name in Centroides.name:
                names_stations.append(name+'_'+rcp+'_'+per[2]+'_Model_'+str(nmodel+1))
            name_met = rcp+'_'+per[2]+'_Model_'+str(nmodel+1)
            
            Serie_Prec = pd.DataFrame(index=pd.date_range(start=str(yearini)+'-01-01 12:00:00',end=str(yearfin)+'-12-31 12:00:00',freq='D'),columns=Centroides.name)
            Serie_Temp_max = pd.DataFrame(index=pd.date_range(start=str(yearini)+'-01-01 12:00:00',end=str(yearfin)+'-12-31 12:00:00',freq='D'),columns=Centroides.name)
            Serie_Temp_min = pd.DataFrame(index=pd.date_range(start=str(yearini)+'-01-01 12:00:00',end=str(yearfin)+'-12-31 12:00:00',freq='D'),columns=Centroides.name)
            Serie_ET = pd.DataFrame(index=pd.date_range(start=str(yearini)+'-01-01 12:00:00',end=str(yearfin)+'-12-31 12:00:00',freq='D'),columns=Centroides.name)
            
            ETP_H = pd.read_csv('E:/TUNEZ/01_Datos_Climaticos/Reconstruccion/'+ 'ETP_Centroides_2.csv',index_col=0, decimal=',',delimiter=';').iloc[:,0::2]
            ETP_T = pd.read_csv('E:/TUNEZ/01_Datos_Climaticos/Reconstruccion/'+ 'ETP_Thorw_Centroides_2.csv',index_col=0,delimiter=';', decimal=',').iloc[:,0::2]
            
            Datos_mensuales = pd.DataFrame(index=np.arange(1,13),columns=Centroides.name)
            
            for c, cc in enumerate(Centroides.name):
                Serie_Prec.loc[:,cc]     = dict_CC_PREC_QM_CORDEX[str(yearini)+'_'+str(yearfin)][rcp][m][cc].values
                Serie_Temp_max.loc[:,cc] = dict_CC_TASMAX_QM_CORDEX[str(yearini)+'_'+str(yearfin)][rcp][m][cc].values
                Serie_Temp_min.loc[:,cc] = dict_CC_TASMIN_QM_CORDEX[str(yearini)+'_'+str(yearfin)][rcp][m][cc].values
                
                tmin = Serie_Temp_min.loc[:,cc]
                tmax = Serie_Temp_max.loc[:,cc]
                tmed = (tmin+tmax)/2
                lat  = Centroides[Centroides.name==cc].loc[:,'POINT_Y'].values
                date = i
                Datos_mensuales.loc[:,cc] = thornthwaite_(tmin.astype(float),tmax.astype(float),lat)
                
            Factor = Datos_mensuales.values/ETP_T.T.values
    
            Serie_Prec.columns = names_stations
            Serie_Prec.to_csv('E:/TUNEZ/07_Cambio_Climatico/Archivo_temporal.csv')
                
            ETP_Oued_Mellah_month=Serie_ET.loc[:,Centroides.name].resample('M').sum()
            ETP_Oued_Mellah_month[ETP_Oued_Mellah_month==0]=np.nan
            
            grouped_med= ETP_Oued_Mellah_month.groupby(lambda x: x.month)
            Datos_mensuales=grouped_med.mean()
            
            Datos_mensuales_table = pd.DataFrame(columns=name_columns,index=Centroides.name)
            Datos_mensuales_table.iloc[:,:] = 2.4
            Datos_mensuales_table.loc[Centroides.name,np.arange(1,13)] = ETP_H.values*Factor.T
            Datos_mensuales_table.columns = Datos_mensuales_table.columns.astype(str)
            
            
                
            Start_Time = '1 January '+str(yearini)+', 00:00' # Fecha de inicio del control
            End_Time   = '31 December '+str(yearfin)+', 00:00'
            
            if sim==0:
                generate_gage(name_model, names_stations, Time_interval, Path_model, Start_Time, End_Time, file_dss,exists_gage=False)
            else:
                generate_gage(name_model, names_stations, Time_interval, Path_model, Start_Time, End_Time, file_dss, exists_gage=True)
            
            fill_gage(names_stations, path_rain, Time_interval, Path_model, file_dss, Start_Time, End_Time)
            generate_met(name_met, Centroides.name,names_stations,Path_model, name_basin,Evapotranspiration=True,ET_Table=Datos_mensuales_table)
            
            
            if sim==0:
                generate_run(Path_model, name_model, 'Run_'+name_met,name_met, name_basin,'Control'+per[2],exists_run=False)
            else:
                generate_run(Path_model, name_model, 'Run_'+name_met,name_met, name_basin,'Control'+per[2],exists_run=True)  
            sim = sim+1

################### El fichero .gage fue modificado satisfactoriamente ###################


100%|██████████████████████████████████████████████████████████████████████████████████| 31/31 [00:09<00:00,  3.27it/s]


################### Se ha modificado el fichero .dss satisfactoriamente y se han llenado los datos de lluvia ###############################
################### Los ficheros .met fueron creados en la carpeta del proyecto satisfactoriamente ###############################
################### El fichero .run fue creado satisfactoriamente ###############################
################### El fichero .gage fue modificado satisfactoriamente ###################


100%|██████████████████████████████████████████████████████████████████████████████████| 31/31 [00:08<00:00,  3.59it/s]


################### Se ha modificado el fichero .dss satisfactoriamente y se han llenado los datos de lluvia ###############################
################### Los ficheros .met fueron creados en la carpeta del proyecto satisfactoriamente ###############################
################### El fichero .run fue creado satisfactoriamente ###############################
################### El fichero .gage fue modificado satisfactoriamente ###################


...

################### Se ha modificado el fichero .dss satisfactoriamente y se han llenado los datos de lluvia ###############################
################### Los ficheros .met fueron creados en la carpeta del proyecto satisfactoriamente ###############################
################### El fichero .run fue creado satisfactoriamente ###############################
################### El fichero .gage fue modificado satisfactoriamente ###################


100%|██████████████████████████████████████████████████████████████████████████████████| 31/31 [00:08<00:00,  3.79it/s]


################### Se ha modificado el fichero .dss satisfactoriamente y se han llenado los datos de lluvia ###############################
################### Los ficheros .met fueron creados en la carpeta del proyecto satisfactoriamente ###############################
################### El fichero .run fue creado satisfactoriamente ###############################
################### El fichero .gage fue modificado satisfactoriamente ###################


100%|██████████████████████████████████████████████████████████████████████████████████| 31/31 [00:08<00:00,  3.59it/s]

################### Se ha modificado el fichero .dss satisfactoriamente y se han llenado los datos de lluvia ###############################
################### Los ficheros .met fueron creados en la carpeta del proyecto satisfactoriamente ###############################
################### El fichero .run fue creado satisfactoriamente ###############################


In [19]:
names_control = read_control(Path_model, file_hms)

In [20]:
names_met =list()
import os
for file in os.listdir(Path_model):
    if file.endswith(".met"):
        names_met.append(file[:-4])

In [21]:
generate_hms(name_model, Path_model, names_met, file_dss, ['Oued_Mellah_SED_CP_RCP45','Oued_Mellah_SED_CP_RCP85',
                                                           'Oued_Mellah_SED_MP_RCP45','Oued_Mellah_SED_MP_RCP85',
                                                           'Oued_Mellah_SED_LP_RCP45','Oued_Mellah_SED_LP_RCP85'], names_control)

################### El fichero .hms fue modificado satisfactoriamente ###############################


In [22]:
for nmodel, m in enumerate(tqdm.tqdm(names_model)):
    for rcp in ['rcp45','rcp85']:
        for per in [[2011,2040,'CP'],[2041,2070,'MP'],[2071,2100,'LP']]: #[2011,2040,'CP'],[2041,2070,'MP'],[2071,2100,'LP']
            print('Simulando '+rcp+' '+per[2]+' '+'Modelo '+ str(nmodel+1))
            Name_run =  'Run_'+rcp+'_'+per[2]+'_Model_'+str(nmodel+1)
            Generate_py(Path_model, name_model, [Name_run.replace("_", " ")])
            
            # Con este modulo se corre el programa HEC HMS, para saber si la modelacion tuvo exito debe aparecer un 0 al final, en caso contrario aparece un -1
            os.chdir('C:/Program Files/HEC/HEC-HMS/4.9/')
            os.system('HEC-HMS.cmd -script '+Path_model+'scripts/compute_current.py')

  0%|                                                                                           | 0/15 [00:00<?, ?it/s]

Simulando rcp45 CP Modelo 1
################### El fichero .py fue creado satisfactoriamente ###############################
Simulando rcp45 MP Modelo 1
################### El fichero .py fue creado satisfactoriamente ###############################
Simulando rcp45 LP Modelo 1
################### El fichero .py fue creado satisfactoriamente ###############################
Simulando rcp85 CP Modelo 1
################### El fichero .py fue creado satisfactoriamente ###############################
Simulando rcp85 MP Modelo 1
################### El fichero .py fue creado satisfactoriamente ###############################
Simulando rcp85 LP Modelo 1
################### El fichero .py fue creado satisfactoriamente ###############################


  7%|█████▌                                                                             | 1/15 [00:31<07:15, 31.12s/it]

Simulando rcp45 CP Modelo 2
################### El fichero .py fue creado satisfactoriamente ###############################
Simulando rcp45 MP Modelo 2
################### El fichero .py fue creado satisfactoriamente ###############################
Simulando rcp45 LP Modelo 2
################### El fichero .py fue creado satisfactoriamente ###############################
Simulando rcp85 CP Modelo 2
################### El fichero .py fue creado satisfactoriamente ###############################
Simulando rcp85 MP Modelo 2
################### El fichero .py fue creado satisfactoriamente ###############################
Simulando rcp85 LP Modelo 2
################### El fichero .py fue creado satisfactoriamente ###############################


 13%|███████████                                                                        | 2/15 [00:59<06:26, 29.74s/it]

...

 87%|███████████████████████████████████████████████████████████████████████           | 13/15 [06:06<00:53, 26.67s/it]

Simulando rcp45 CP Modelo 14
################### El fichero .py fue creado satisfactoriamente ###############################
Simulando rcp45 MP Modelo 14
################### El fichero .py fue creado satisfactoriamente ###############################
Simulando rcp45 LP Modelo 14
################### El fichero .py fue creado satisfactoriamente ###############################
Simulando rcp85 CP Modelo 14
################### El fichero .py fue creado satisfactoriamente ###############################
Simulando rcp85 MP Modelo 14
################### El fichero .py fue creado satisfactoriamente ###############################
Simulando rcp85 LP Modelo 14
################### El fichero .py fue creado satisfactoriamente ###############################


 93%|████████████████████████████████████████████████████████████████████████████▌     | 14/15 [06:31<00:26, 26.28s/it]

Simulando rcp45 CP Modelo 15
################### El fichero .py fue creado satisfactoriamente ###############################
Simulando rcp45 MP Modelo 15
################### El fichero .py fue creado satisfactoriamente ###############################
Simulando rcp45 LP Modelo 15
################### El fichero .py fue creado satisfactoriamente ###############################
Simulando rcp85 CP Modelo 15
################### El fichero .py fue creado satisfactoriamente ###############################
Simulando rcp85 MP Modelo 15
################### El fichero .py fue creado satisfactoriamente ###############################
Simulando rcp85 LP Modelo 15
################### El fichero .py fue creado satisfactoriamente ###############################


100%|██████████████████████████████████████████████████████████████████████████████████| 15/15 [06:57<00:00, 27.82s/it]


## Extraer resultados Modelo Embalse

In [ ]:
Path_model      = 'E:/TUNEZ/05_Model/Oued_Mellah_CC_DAM/'
for nmodel, m in enumerate(tqdm.tqdm(names_model)):
    for rcp in ['rcp45','rcp85']:
        for per in [[2011,2040,'CP'],[2041,2070,'MP'],[2071,2100,'LP']]:
            if os.path.exists('E:/TUNEZ/07_Cambio_Climatico/CAUDAL/INFLOW_RESERVOIR_2_'+rcp+'_'+per[2]+'_Model_'+m+'_2.csv'):
                continue
            else:
                try:
                    print(m + ' '+ rcp+' '+per[2])
                    Name_run =  'Run_'+rcp+'_'+per[2]+'_Model_'+str(nmodel+1)
                    fid = HecDss.Open(Path_model+Name_run+'.dss')
                    
                    pn = '//JUNT7/FLOW/01JAN'+str(per[0])+'/1HOUR/RUN:'+Name_run.replace("_", " ")
                    ts = fid.read_ts(pn,window=(str(per[0])+'-01-01',str(per[1]+1)+'-12-31'))
                    values = pd.DataFrame(ts.values)
                    times = np.array(ts.pytimes)
                    Q_OUCH= pd.DataFrame(index= times, columns = ['flow'])
                    Q_OUCH.loc[:,'flow'] = values.iloc[:,0].values


                    pn = '//RESERVOIR-1-SPILL-1/FLOW/01JAN'+str(per[0])+'/1HOUR/RUN:'+Name_run.replace("_", " ")
                    ts = fid.read_ts(pn,window=(str(per[0])+'-01-01',str(per[1]+1)+'-12-31'))
                    values = pd.DataFrame(ts.values)
                    times = np.array(ts.pytimes)
                    Q_SPILL = pd.DataFrame(index= times, columns = ['m3/s'])
                    Q_SPILL.loc[:,'m3/s'] = values.iloc[:,0].values

                    del values

                    pn_ele = '//RESERVOIR-1/ELEVATION/01JAN'+str(per[0])+'/1HOUR/RUN:'+Name_run.replace("_", " ")
                    ts_elev = fid.read_ts(pn_ele,window=(str(per[0])+'-01-01',str(per[1]+1)+'-12-31'))
                    values = pd.DataFrame(ts_elev.values)
                    times = np.array(ts_elev.pytimes)
                    ELEV = pd.DataFrame(index= times, columns = ['msnm'])
                    ELEV.loc[:,'msnm'] = values.iloc[:,0].values

                    del values

                    pn_stor = '//RESERVOIR-1/STORAGE/01JAN'+str(per[0])+'/1HOUR/RUN:'+Name_run.replace("_", " ")
                    ts_stor = fid.read_ts(pn_stor,window=(str(per[0])+'-01-01',str(per[1]+1)+'-12-31'))
                    values = pd.DataFrame(ts_stor.values)
                    times = np.array(ts_stor.pytimes)
                    STOR = pd.DataFrame(index= times, columns = ['Hm3'])
                    STOR.loc[:,'Hm3'] = values.iloc[:,0].values*10**-3

                    del values

                    pn_out_reach = '//REACH19/FLOW/01JAN'+str(per[0])+'/1HOUR/RUN:'+Name_run.replace("_", " ")
                    ts_out_reach = fid.read_ts(pn_out_reach,window=(str(per[0])+'-01-01',str(per[1]+1)+'-12-31'))
                    values_rch = pd.DataFrame(ts_out_reach.values)
                    times = np.array(ts_out_reach.pytimes)

                    pn_out_bas = '//BASIN93/FLOW/01JAN'+str(per[0])+'/1HOUR/RUN:'+Name_run.replace("_", " ")
                    ts_out_bas = fid.read_ts(pn_out_bas,window=(str(per[0])+'-01-01',str(per[1]+1)+'-12-31'))
                    values_bas = pd.DataFrame(ts_out_bas.values)
                    times = np.array(ts_out_bas.pytimes)

                    INFLOW = pd.DataFrame(index= times, columns = ['m3/s'])
                    INFLOW.loc[:,'m3/s'] = values_bas.iloc[:,0].values+values_rch.iloc[:,0].values

                    pn_dema = '//SEJNANE/FLOW-DIVERSION/01JAN'+str(per[0])+'/1HOUR/RUN:'+Name_run.replace("_", " ")
                    ts_dema = fid.read_ts(pn_dema,window=(str(per[0])+'-01-01',str(per[1]+1)+'-12-31'))
                    values = pd.DataFrame(ts_dema.values)
                    times = np.array(ts_dema.pytimes)
                    DEMA = pd.DataFrame(index= times, columns = ['Hm3'])
                    DEMA.loc[:,'Hm3'] = values.iloc[:,0].values


                    Q_SPILL.to_csv('E:/TUNEZ/07_Cambio_Climatico/CAUDAL/SPILL_RESERVOIR_2_'+rcp+'_'+per[2]+'_Model_'+m+'_2.csv')
                    ELEV.to_csv('E:/TUNEZ/07_Cambio_Climatico/CAUDAL/ELEV_RESERVOIR_2_'+rcp+'_'+per[2]+'_Model_'+m+'_2.csv')
                    STOR.to_csv('E:/TUNEZ/07_Cambio_Climatico/CAUDAL/STORAGE_RESERVOIR_2_'+rcp+'_'+per[2]+'_Model_'+m+'_2.csv')
                    INFLOW.to_csv('E:/TUNEZ/07_Cambio_Climatico/CAUDAL/INFLOW_RESERVOIR_2_'+rcp+'_'+per[2]+'_Model_'+m+'_2.csv')
                    DEMA.to_csv('E:/TUNEZ/07_Cambio_Climatico/CAUDAL/DEMAND_RESERVOIR_2_'+rcp+'_'+per[2]+'_Model_'+m+'_2.csv')
                    Q_OUCH.to_csv('E:/TUNEZ/07_Cambio_Climatico/CAUDAL/INFLOW_OUCHTATA_'+rcp+'_'+per[2]+'_Model_'+m+'_2.csv')
                except:
                    continue
                fid.close()

  0%|                                                                                           | 0/15 [00:00<?, ?it/s]

CNRM-CERFACS-CNRM-CM5_SMHI-RCA4_r1i1p1 rcp45 CP
CNRM-CERFACS-CNRM-CM5_SMHI-RCA4_r1i1p1 rcp45 MP


In [ ]:
Path_model      = 'E:/TUNEZ/05_Model/Oued_Mellah_CAL_DAM/'
Name_run =  'Hist_1979-2005'
fid = HecDss.Open(Path_model+'Hist_1979_2005'+'.dss')

per = [1979,2005]

pn = '//JUNT7/FLOW/01JAN'+str(per[0])+'/1HOUR/RUN:'+Name_run.replace("_", " ")
ts = fid.read_ts(pn,window=(str(per[0])+'-01-01',str(per[1]+1)+'-12-31'))
values = pd.DataFrame(ts.values)
times = np.array(ts.pytimes)
Q_OUCH= pd.DataFrame(index= times, columns = ['flow'])
Q_OUCH.loc[:,'flow'] = values.iloc[:,0].values

pn = '//RESERVOIR-1-SPILL-1/FLOW/01JAN'+str(per[0])+'/1HOUR/RUN:'+Name_run.upper().replace("_", " ")
ts = fid.read_ts(pn,window=(str(per[0])+'-01-01',str(per[1]+1)+'-12-31'))
values = pd.DataFrame(ts.values)
times = np.array(ts.pytimes)
Q_SPILL = pd.DataFrame(index= times, columns = ['m3/s'])
Q_SPILL.loc[:,'m3/s'] = values.iloc[:,0].values

del values

pn_ele = '//RESERVOIR-1/ELEVATION/01JAN'+str(per[0])+'/1HOUR/RUN:'+Name_run.upper().replace("_", " ")
ts_elev = fid.read_ts(pn_ele,window=(str(per[0])+'-01-01',str(per[1]+1)+'-12-31'))
values = pd.DataFrame(ts_elev.values)
times = np.array(ts_elev.pytimes)
ELEV = pd.DataFrame(index= times, columns = ['msnm'])
ELEV.loc[:,'msnm'] = values.iloc[:,0].values

del values

pn_stor = '//RESERVOIR-1/STORAGE/01JAN'+str(per[0])+'/1HOUR/RUN:'+Name_run.upper().replace("_", " ")
ts_stor = fid.read_ts(pn_stor,window=(str(per[0])+'-01-01',str(per[1]+1)+'-12-31'))
values = pd.DataFrame(ts_stor.values)
times = np.array(ts_stor.pytimes)
STOR = pd.DataFrame(index= times, columns = ['Hm3'])
STOR.loc[:,'Hm3'] = values.iloc[:,0].values*10**-3

del values

pn_out_reach = '//REACH19/FLOW/01JAN'+str(per[0])+'/1HOUR/RUN:'+Name_run.upper().replace("_", " ")
ts_out_reach = fid.read_ts(pn_out_reach,window=(str(per[0])+'-01-01',str(per[1]+1)+'-12-31'))
values_rch = pd.DataFrame(ts_out_reach.values)
times = np.array(ts_out_reach.pytimes)

pn_out_bas = '//BASIN93/FLOW/01JAN'+str(per[0])+'/1HOUR/RUN:'+Name_run.upper().replace("_", " ")
ts_out_bas = fid.read_ts(pn_out_bas,window=(str(per[0])+'-01-01',str(per[1]+1)+'-12-31'))
values_bas = pd.DataFrame(ts_out_bas.values)
times = np.array(ts_out_bas.pytimes)

INFLOW = pd.DataFrame(index= times, columns = ['m3/s'])
INFLOW.loc[:,'m3/s'] = values_bas.iloc[:,0].values+values_rch.iloc[:,0].values

pn_dema = '//SEJNANE/FLOW-DIVERSION/01JAN'+str(per[0])+'/1HOUR/RUN:'+Name_run.upper().replace("_", " ")
ts_dema = fid.read_ts(pn_dema,window=(str(per[0])+'-01-01',str(per[1]+1)+'-12-31'))
values = pd.DataFrame(ts_dema.values)
times = np.array(ts_dema.pytimes)
DEMA = pd.DataFrame(index= times, columns = ['Hm3'])
DEMA.loc[:,'Hm3'] = values.iloc[:,0].values

Q_SPILL.columns = ['m3/s']
ELEV.columns = ['msnm']
STOR.columns = ['Hm3']
INFLOW.columns = ['m3/s']
DEMA.columns = ['m3/s']


Q_SPILL.resample('D').mean().to_excel('E:/TUNEZ/07_Cambio_Climatico/GESTION/SPILL_RESERVOIR_historical_2.xlsx')
ELEV.resample('D').mean().to_excel('E:/TUNEZ/07_Cambio_Climatico/GESTION/ELEV_RESERVOIR_historical_2.xlsx')
STOR.resample('D').mean().to_excel('E:/TUNEZ/07_Cambio_Climatico/GESTION/STORAGE_RESERVOIR_historical_2.xlsx')
INFLOW.resample('D').mean().to_excel('E:/TUNEZ/07_Cambio_Climatico/GESTION/INFLOW_RESERVOIR_historical_2.xlsx')
DEMA.resample('D').mean().to_excel('E:/TUNEZ/07_Cambio_Climatico/GESTION/DEMAND_RESERVOIR_historical_2.xlsx')
Q_OUCH.resample('D').mean().to_excel('E:/TUNEZ/07_Cambio_Climatico/GESTION/INFLOW_OUCHTATA_historical_2.xlsx')

               
fid.close()

In [ ]:
for rcp in ['rcp45','rcp85']:
    for per in [[2011,2040,'CP'],[2041,2070,'MP'],[2071,2100,'LP']]:
        n = 0
        for nmodel, m in enumerate(tqdm.tqdm(names_model)):
            if n==0:
                try:
                    Q_SPILL = pd.read_csv('E:/TUNEZ/07_Cambio_Climatico/CAUDAL/SPILL_RESERVOIR_2_'+rcp+'_'+per[2]+'_Model_'+m+'_2.csv',index_col=0,parse_dates=True)
                    ELEV = pd.read_csv('E:/TUNEZ/07_Cambio_Climatico/CAUDAL/ELEV_RESERVOIR_2_'+rcp+'_'+per[2]+'_Model_'+m+'_2.csv',index_col=0,parse_dates=True)
                    STOR = pd.read_csv('E:/TUNEZ/07_Cambio_Climatico/CAUDAL/STORAGE_RESERVOIR_2_'+rcp+'_'+per[2]+'_Model_'+m+'_2.csv',index_col=0,parse_dates=True)
                    INFLOW = pd.read_csv('E:/TUNEZ/07_Cambio_Climatico/CAUDAL/INFLOW_RESERVOIR_2_'+rcp+'_'+per[2]+'_Model_'+m+'_2.csv',index_col=0,parse_dates=True)
                    DEMA = pd.read_csv('E:/TUNEZ/07_Cambio_Climatico/CAUDAL/DEMAND_RESERVOIR_2_'+rcp+'_'+per[2]+'_Model_'+m+'_2.csv',index_col=0,parse_dates=True)
                    OUCH = pd.read_csv('E:/TUNEZ/07_Cambio_Climatico/CAUDAL/INFLOW_OUCHTATA_'+rcp+'_'+per[2]+'_Model_'+m+'_2.csv',index_col=0,parse_dates=True)
                    n = n+1
                except:
                    continue
            else:
                try:
                    Q_SPILL_2 = pd.read_csv('E:/TUNEZ/07_Cambio_Climatico/CAUDAL/SPILL_RESERVOIR_2_'+rcp+'_'+per[2]+'_Model_'+m+'_2.csv',index_col=0,parse_dates=True)
                    ELEV_2 = pd.read_csv('E:/TUNEZ/07_Cambio_Climatico/CAUDAL/ELEV_RESERVOIR_2_'+rcp+'_'+per[2]+'_Model_'+m+'_2.csv',index_col=0,parse_dates=True)
                    STOR_2 = pd.read_csv('E:/TUNEZ/07_Cambio_Climatico/CAUDAL/STORAGE_RESERVOIR_2_'+rcp+'_'+per[2]+'_Model_'+m+'_2.csv',index_col=0,parse_dates=True)
                    INFLOW_2 = pd.read_csv('E:/TUNEZ/07_Cambio_Climatico/CAUDAL/INFLOW_RESERVOIR_2_'+rcp+'_'+per[2]+'_Model_'+m+'_2.csv',index_col=0,parse_dates=True)
                    DEMA_2 = pd.read_csv('E:/TUNEZ/07_Cambio_Climatico/CAUDAL/DEMAND_RESERVOIR_2_'+rcp+'_'+per[2]+'_Model_'+m+'_2.csv',index_col=0,parse_dates=True)
                    OUCH_2 = pd.read_csv('E:/TUNEZ/07_Cambio_Climatico/CAUDAL/INFLOW_OUCHTATA_'+rcp+'_'+per[2]+'_Model_'+m+'_2.csv',index_col=0,parse_dates=True)

                    Q_SPILL = pd.concat((Q_SPILL,Q_SPILL_2),axis=1)
                    ELEV = pd.concat((ELEV,ELEV_2),axis=1)
                    STOR = pd.concat((Q_SPILL,STOR_2),axis=1)
                    INFLOW = pd.concat((INFLOW,INFLOW_2),axis=1)
                    DEMA = pd.concat((DEMA,DEMA_2),axis=1)
                    OUCH = pd.concat((OUCH,OUCH_2),axis=1)
                    n = n+1
                except:
                    continue
                
        Q_SPILL_ense = pd.DataFrame(Q_SPILL.median(axis=1))
        ELEV_ense    = pd.DataFrame(ELEV.median(axis=1))
        STOR_ense    = pd.DataFrame(STOR.median(axis=1))
        INFLOW_ense  = pd.DataFrame(INFLOW.median(axis=1))
        DEMA_ense    = pd.DataFrame(DEMA.median(axis=1))
        OUCH_ense    = pd.DataFrame(OUCH.median(axis=1))
        
        Q_SPILL_ense.columns = ['m3/s']
        ELEV_ense.columns = ['msnm']
        STOR_ense.columns = ['Hm3']
        INFLOW_ense.columns = ['m3/s']
        DEMA_ense.columns = ['m3/s']
        OUCH_ense.columns = ['m3/s']
        
        
        Q_SPILL_ense.resample('D').mean().to_excel('E:/TUNEZ/07_Cambio_Climatico/GESTION/SPILL_RESERVOIR_'+rcp+'_'+per[2]+'_Ensemble'+'_2.xlsx')
        ELEV_ense.resample('D').mean().to_excel('E:/TUNEZ/07_Cambio_Climatico/GESTION/ELEV_RESERVOIR_'+rcp+'_'+per[2]+'_Ensemble'+'_2.xlsx')
        STOR_ense.resample('D').mean().to_excel('E:/TUNEZ/07_Cambio_Climatico/GESTION/STORAGE_RESERVOIR_'+rcp+'_'+per[2]+'_Ensemble'+'_2.xlsx')
        INFLOW_ense.resample('D').mean().to_excel('E:/TUNEZ/07_Cambio_Climatico/GESTION/INFLOW_RESERVOIR_'+rcp+'_'+per[2]+'_Ensemble'+'_2.xlsx')
        DEMA_ense.resample('D').mean().to_excel('E:/TUNEZ/07_Cambio_Climatico/GESTION/DEMAND_RESERVOIR_'+rcp+'_'+per[2]+'_Ensemble'+'_2.xlsx')
        OUCH_ense.resample('D').mean().to_excel('E:/TUNEZ/07_Cambio_Climatico/GESTION/INFLOW_OUCHTATA_'+rcp+'_'+per[2]+'_Ensemble'+'_2.xlsx')